# STL Schritt fuer Schritt im CRISP-DM-Prozess

Dieses Notebook erklaert die **STL-Zerlegung** didaktisch und Schritt fuer Schritt.

STL steht fuer **Seasonal and Trend decomposition using LOESS**. Die Methode zerlegt eine Zeitreihe in drei Bestandteile:

\[
y_t = T_t + S_t + R_t
\]

- \(T_t\): Trend, also die langfristige Entwicklung
- \(S_t\): saisonaler Effekt, also ein wiederkehrendes Muster
- \(R_t\): Residuum, also der nicht erklaerte Rest

Die Reihenfolge orientiert sich am **CRISP-DM-Modell**:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

Wichtig: CRISP-DM ist kein einmaliger Durchlauf. Feedback aus der Evaluation oder aus dem Fachbereich kann dazu fuehren, dass wir eine weitere Iteration starten und Annahmen, Daten oder Modellierung anpassen.

## 1. Business Understanding

Wir stellen uns ein Unternehmen vor, das monatliche Nachfragewerte analysieren moechte.

Die fachlichen Fragen lauten:

- Waechst die Nachfrage langfristig?
- Gibt es ein stabiles saisonales Muster?
- Welche Monate sind ungewoehnlich und sollten fachlich geprueft werden?
- Ist ein additives Modell ausreichend, oder waechst die Saisonalitaet mit dem Niveau der Zeitreihe?

STL hilft hier nicht direkt als Prognosemodell, sondern als **Erklaerungs- und Vorverarbeitungsschritt**. Es macht sichtbar, welche Struktur in der Zeitreihe steckt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

rng = np.random.default_rng(42)

## 2. Data Understanding

Fuer die Erklaerung verwenden wir zuerst eine kuenstliche monatliche Zeitreihe.

Der Vorteil: Wir kennen die ungefaehre Struktur der Daten. Dadurch kann man besser sehen, was STL rekonstruiert:

- ein langsam steigender Trend
- ein jaehrliches saisonales Muster
- zufaellige Schwankungen
- einzelne Ausreisser

Die Ausreisser sind absichtlich eingebaut, damit spaeter der Unterschied zwischen **standard STL** und **robust STL** sichtbar wird.

In [ ]:
dates = pd.date_range("2018-01-01", periods=72, freq="MS")
t = np.arange(len(dates))

trend = 120 + 1.4 * t
seasonal_pattern = np.array([-18, -12, 4, 10, 14, 22, 30, 26, 12, 2, -10, -20])
seasonal = np.tile(seasonal_pattern, len(dates) // 12)
noise = rng.normal(0, 5, size=len(dates))

sales = trend + seasonal + noise
sales[[22, 45]] += [55, -48]

ts = pd.Series(sales, index=dates, name="nachfrage")
ts.head()

In [ ]:
ax = ts.plot(title="Monatliche Nachfrage mit Trend, Saisonalitaet und Ausreissern")
ax.set_xlabel("Datum")
ax.set_ylabel("Nachfrage")
plt.show()

### Erste Beobachtung

Schon vor der Modellierung erkennt man drei Dinge:

- Die Reihe steigt langfristig.
- Das Muster wiederholt sich ungefaehr alle 12 Monate.
- Einige Punkte wirken ungewoehnlich hoch oder niedrig.

Diese Beobachtungen fuehren direkt zur Wahl von `period=12`, weil die Daten monatlich sind und ein Jahr aus 12 Monaten besteht.

## 3. Data Preparation

STL erwartet eine geordnete Zeitreihe mit regelmaessiger Frequenz.

In echten Projekten wuerde man hier fehlende Werte, doppelte Zeitpunkte, Ausreisser, Kalendereffekte und Aggregationslogik pruefen. In diesem Beispiel sind die Daten bereits monatlich und vollstaendig.

In [ ]:
pruefung = pd.DataFrame({
    "anzahl_werte": [ts.shape[0]],
    "fehlende_werte": [ts.isna().sum()],
    "start": [ts.index.min()],
    "ende": [ts.index.max()],
    "frequenz": [pd.infer_freq(ts.index)],
})

pruefung

## 4. Modeling: Standard STL

Jetzt wenden wir STL an.

Die wichtigsten Einstellungen sind:

- `period=12`: Das saisonale Muster wiederholt sich alle 12 Monate.
- `robust=False`: Standard STL behandelt alle Beobachtungen gleich, auch Ausreisser.

Das Ergebnis besteht aus beobachteter Reihe, Trend, Saison und Residuum.

In [ ]:
standard_result = STL(ts, period=12, robust=False).fit()

fig = standard_result.plot()
fig.set_size_inches(12, 8)
plt.suptitle("Standard STL", y=1.02)
plt.show()

In [ ]:
standard_components = pd.DataFrame({
    "beobachtet": ts,
    "trend": standard_result.trend,
    "saisonal": standard_result.seasonal,
    "residuum": standard_result.resid,
})

standard_components.head(15)

### Komponenten lesen

Die Zerlegung ist eine additive Rekonstruktion:

\[
beobachtet \approx trend + saisonal + residuum
\]

Der Trend beschreibt die langsame Grundbewegung. Die saisonale Komponente beschreibt typische Monatsabweichungen. Das Residuum zeigt, was nach Trend und Saison uebrig bleibt.

In [ ]:
rekonstruktion = standard_components["trend"] + standard_components["saisonal"] + standard_components["residuum"]
maximaler_fehler = (standard_components["beobachtet"] - rekonstruktion).abs().max()

print(f"Maximaler Rekonstruktionsfehler: {maximaler_fehler:.10f}")

## 5. Evaluation

In CRISP-DM pruefen wir jetzt, ob die Zerlegung fachlich sinnvoll ist.

Typische Fragen:

- Ist der Trend plausibel glatt?
- Ist die Saison stabil und interpretierbar?
- Sind grosse Residuen fachlich erklaerbar?
- Stoeren Ausreisser die Trend- oder Saisonkomponente?

Diese Evaluation ist bewusst nicht nur technisch. Sie sollte mit Feedback aus dem Fachbereich verbunden werden.

In [ ]:
residuen = standard_components["residuum"]
grenze = 2 * residuen.std()

auffaellig = standard_components.loc[residuen.abs() > grenze, ["beobachtet", "trend", "saisonal", "residuum"]]
auffaellig

In [ ]:
ax = residuen.plot(title="Residuen aus Standard STL")
ax.axhline(grenze, linestyle="--", color="tab:red", label="+/- 2 Standardabweichungen")
ax.axhline(-grenze, linestyle="--", color="tab:red")
ax.set_xlabel("Datum")
ax.set_ylabel("Residuum")
ax.legend()
plt.show()

## Feedback und zweite CRISP-DM-Iteration

Angenommen, der Fachbereich sagt:

> Die auffaelligen Monate waren Sonderaktionen und Lieferprobleme. Sie sollen den normalen Trend moeglichst wenig beeinflussen.

Dieses Feedback fuehrt zu einer weiteren Iteration:

- Business Understanding wird geschaerft: Gesucht ist das normale Muster ohne starke Ausreisserwirkung.
- Data Understanding wird erweitert: Die auffaelligen Monate sind Sonderereignisse.
- Modeling wird angepasst: Wir verwenden **robust STL**.

Das ist typisch fuer CRISP-DM: Die Evaluation fuehrt nicht nur zum Abschluss, sondern oft zur Verbesserung des naechsten Durchlaufs.

## 6. Modeling: Robust STL im Vergleich

Robust STL reduziert den Einfluss von Ausreissern auf Trend und saisonale Komponente.

In `statsmodels` geschieht das mit:

```python
STL(..., robust=True)
```

Die Ausreisser verschwinden dadurch nicht. Sie landen aber staerker im Residuum, statt Trend und Saison zu verzerren.

In [ ]:
robust_result = STL(ts, period=12, robust=True).fit()

comparison = pd.DataFrame({
    "beobachtet": ts,
    "trend_standard": standard_result.trend,
    "trend_robust": robust_result.trend,
    "residuum_standard": standard_result.resid,
    "residuum_robust": robust_result.resid,
})

comparison.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

comparison["beobachtet"].plot(ax=axes[0], title="Beobachtete Zeitreihe")
axes[0].set_ylabel("Nachfrage")

comparison[["trend_standard", "trend_robust"]].plot(ax=axes[1], title="Trend: Standard STL vs. Robust STL")
axes[1].set_ylabel("Trend")

comparison[["residuum_standard", "residuum_robust"]].plot(ax=axes[2], title="Residuen: Standard STL vs. Robust STL")
axes[2].set_ylabel("Residuum")
axes[2].set_xlabel("Datum")

plt.tight_layout()
plt.show()

In [ ]:
vergleich_ausreisser = comparison.loc[comparison["residuum_robust"].abs().nlargest(5).index]
vergleich_ausreisser.sort_index()

### Interpretation des robusten Vergleichs

Bei starken Sonderereignissen ist robust STL oft besser geeignet, wenn das Ziel ein stabiles Normalmuster ist.

Standard STL kann Ausreisser staerker in Trend oder Saison aufnehmen. Robust STL laesst solche Punkte eher im Residuum sichtbar. Das ist fuer fachliche Kommunikation hilfreich: Man kann sagen, welche Monate nicht zum normalen Muster passen.

## 7. Deployment: Was wuerde man mit dem Ergebnis tun?

In einem echten Projekt koennte man die Komponenten nutzen, um:

- saisonbereinigte Nachfrage zu berichten
- auffaellige Monate fuer Ursachenanalyse zu markieren
- ein Prognosemodell auf der saisonbereinigten Reihe aufzubauen
- typische saisonale Faktoren in Planung und Budgetierung zu verwenden

Deployment bedeutet hier nicht zwingend eine produktive Software. Es kann auch ein wiederholbarer Analysebericht, ein Dashboard oder ein dokumentierter Prognoseprozess sein.

## 8. Multiplikative Saisonalitaet

Bisher war die Zerlegung additiv:

\[
y_t = T_t + S_t + R_t
\]

Das passt gut, wenn die saisonale Schwankung ungefaehr gleich gross bleibt.

Manchmal wird die saisonale Schwankung aber groesser, wenn das Niveau der Zeitreihe steigt. Dann ist ein multiplikatives Denken passender:

\[
y_t = T_t \times S_t \times R_t
\]

Klassisches STL in `statsmodels` ist additiv. Eine haeufige Loesung ist deshalb die **Log-Transformation**:

\[
\log(y_t) = \log(T_t) + \log(S_t) + \log(R_t)
\]

Auf der Log-Skala wird ein multiplikatives Muster also additiv behandelbar.

In [ ]:
dates_mult = pd.date_range("2016-01-01", periods=96, freq="MS")
t_mult = np.arange(len(dates_mult))

level = 80 * np.exp(0.012 * t_mult)
seasonal_multiplier_pattern = np.array([0.82, 0.88, 0.96, 1.02, 1.07, 1.14, 1.24, 1.20, 1.08, 1.00, 0.91, 0.84])
seasonal_multiplier = np.tile(seasonal_multiplier_pattern, len(dates_mult) // 12)
noise_multiplier = rng.lognormal(mean=0, sigma=0.035, size=len(dates_mult))

mult_ts = pd.Series(level * seasonal_multiplier * noise_multiplier, index=dates_mult, name="umsatz")

ax = mult_ts.plot(title="Multiplikative Zeitreihe: Saisonale Ausschlaege wachsen mit dem Niveau")
ax.set_xlabel("Datum")
ax.set_ylabel("Umsatz")
plt.show()

### Additives STL direkt auf der multiplikativen Reihe

Wenn man additives STL direkt auf diese Reihe anwendet, wird die Saisonkomponente in absoluten Einheiten geschaetzt.

Das kann zwar funktionieren, aber die saisonalen Ausschlaege sind am Ende der Reihe groesser als am Anfang. Eine einzige additive Saisonkurve beschreibt dieses Verhalten oft weniger elegant.

In [ ]:
mult_additive = STL(mult_ts, period=12, robust=True).fit()

fig = mult_additive.plot()
fig.set_size_inches(12, 8)
plt.suptitle("Additives STL direkt auf der multiplikativen Reihe", y=1.02)
plt.show()

### Log-Transformation fuer multiplikative Muster

Jetzt transformieren wir die Reihe mit dem natuerlichen Logarithmus und wenden STL auf `log(y)` an.

Danach koennen Trend und saisonbereinigte Werte wieder mit `np.exp(...)` auf die urspruengliche Skala gebracht werden.

In [ ]:
log_mult_ts = np.log(mult_ts)
log_result = STL(log_mult_ts, period=12, robust=True).fit()

log_components = pd.DataFrame({
    "beobachtet": mult_ts,
    "log_beobachtet": log_mult_ts,
    "log_trend": log_result.trend,
    "log_saisonal": log_result.seasonal,
    "log_residuum": log_result.resid,
})

log_components["trend_originalskala"] = np.exp(log_components["log_trend"])
log_components["saisonfaktor"] = np.exp(log_components["log_saisonal"])
log_components["saisonbereinigt_originalskala"] = np.exp(
    log_components["log_beobachtet"] - log_components["log_saisonal"]
)

log_components.head(15)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

mult_ts.plot(ax=axes[0], title="Originale multiplikative Reihe")
axes[0].set_ylabel("Umsatz")

log_components["trend_originalskala"].plot(ax=axes[1], title="Trend aus Log-STL zurueck auf Originalskala")
axes[1].set_ylabel("Trend")

log_components["saisonfaktor"].plot(ax=axes[2], title="Saisonfaktor aus Log-STL")
axes[2].axhline(1.0, color="black", linewidth=1)
axes[2].set_ylabel("Faktor")
axes[2].set_xlabel("Datum")

plt.tight_layout()
plt.show()

In [ ]:
monat_namen = {
    1: "Januar", 2: "Februar", 3: "Maerz", 4: "April",
    5: "Mai", 6: "Juni", 7: "Juli", 8: "August",
    9: "September", 10: "Oktober", 11: "November", 12: "Dezember",
}

saisonfaktoren = log_components.assign(monat=log_components.index.month.map(monat_namen))
mittlere_saisonfaktoren = saisonfaktoren.groupby("monat")["saisonfaktor"].mean().reindex(list(monat_namen.values()))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(mittlere_saisonfaktoren.index, mittlere_saisonfaktoren.values)
ax.set_title("Durchschnittlicher multiplikativer Saisonfaktor nach Monat")
ax.axhline(1.0, color="black", linewidth=1)
ax.set_xlabel("Monat")
ax.set_ylabel("Faktor")
plt.xticks(rotation=45)
plt.show()

mittlere_saisonfaktoren

### Interpretation der Log-Transformation

Ein Saisonfaktor von `1.20` bedeutet: In diesem Monat liegt die Nachfrage typischerweise etwa 20 Prozent ueber dem Trendniveau.

Ein Faktor von `0.85` bedeutet: In diesem Monat liegt die Nachfrage typischerweise etwa 15 Prozent unter dem Trendniveau.

Die Log-Transformation ist deshalb hilfreich, wenn die Saisonalitaet nicht als konstanter absoluter Effekt, sondern als relativer Prozent- oder Faktor-Effekt verstanden werden soll.

## Zusammenfassung

Dieses Notebook hat STL im CRISP-DM-Rahmen erklaert:

- Im Business Understanding wurde geklaert, welche fachlichen Fragen STL beantworten soll.
- Im Data Understanding wurde die Zeitreihe visuell und strukturell geprueft.
- In der Data Preparation wurde sichergestellt, dass eine regelmaessige monatliche Reihe vorliegt.
- Im Modeling wurde zuerst Standard STL angewendet.
- In der Evaluation wurden Residuen und Ausreisser bewertet.
- Durch Feedback wurde eine zweite Iteration mit robust STL begruendet.
- Fuer multiplikative Muster wurde gezeigt, warum die Log-Transformation sinnvoll ist.

Die wichtigste Idee lautet: STL ist nicht nur eine Formel. STL ist ein Werkzeug, um Zeitreihen fachlich verstaendlich zu machen und bessere naechste Modellierungsschritte vorzubereiten.